In [1]:
import numpy as np
import openfhe_numpy as onp
from openfhe import *

In [2]:
def slice_vector_he(enc_vec, start, end, keys, debug=False):
    """
    Slice an encrypted vector while keeping it encrypted using matrix multiplication.
    
    Treat the vector of length n as a matrix of shape (1, n).
    Create a selection mask in numpy which is 0 everywhere except for the range [start, end)
    Matrix multiply the selection matrix with the original vector to get the sliced result. 
    
    Parameters
    ----------
    enc_vec :  CTArray
        The encrypted vector to slice (in ROW_MAJOR format)
    start : int
        Start index (inclusive)
    end : int
        End index (exclusive)
    keys:
        The keys generated by the cryptocontext (cc.KeyGen())
    
    Returns
    -------
    CTArray
        The sliced encrypted vector containing elements from index start to end-1
    """
    slice_length = end - start
    length = enc_vec.original_shape[0]

    selection_matrix = np.zeros((slice_length, length))
    for i in range(slice_length):
        selection_matrix[i, start + i] = 1.0
    
    if debug:
        import pdb; pdb.set_trace()

    enc_selection = onp.array(
        cc=enc_vec.data.GetCryptoContext(),
        data=selection_matrix,
        batch_size=enc_vec.batch_size,
        order=onp.ROW_MAJOR,
        mode="tile",
        fhe_type="C",
        public_key=keys.publicKey,
    )
    enc_selection.extra["colkey"] = onp.sum_col_keys(keys.secretKey) # maybe this can be moved outside so only public key would be needed in this function

    slice_result = enc_selection @ enc_vec

    return slice_result

In [11]:
def encrypt(data, crypto_context, keys, mode="ROW_MAJOR"):
    """
    Here data is encrypted using the crypto_context and the keys.
    We are using 'tile' packing here
    """

    batch_size = crypto_context.GetRingDimension() // 2

    if mode == "ROW_MAJOR":
        mode = onp.ROW_MAJOR
    else:
        mode = onp.COL_MAJOR

    enc_data = onp.array(
        cc=crypto_context,
        data=data,
        batch_size=batch_size,
        order=mode,
        mode="tile",
        fhe_type="C",
        public_key=keys.publicKey
    )

    return enc_data

In [12]:
mult_depth = 10

params = CCParamsCKKSRNS()
params.SetMultiplicativeDepth(mult_depth)
params.SetScalingModSize(59)
params.SetFirstModSize(60)
params.SetScalingTechnique(FIXEDAUTO)
params.SetKeySwitchTechnique(HYBRID)
params.SetSecretKeyDist(UNIFORM_TERNARY)

cc = GenCryptoContext(params)
cc.Enable(PKESchemeFeature.PKE)
cc.Enable(PKESchemeFeature.LEVELEDSHE)
cc.Enable(PKESchemeFeature.ADVANCEDSHE)

keys = cc.KeyGen()

cc.EvalMultKeyGen(keys.secretKey)
cc.EvalSumKeyGen(keys.secretKey)

In [28]:
# create a noise vector
num_samples = 4096
noise = np.random.randn(num_samples)
noise.shape

(4096,)

In [29]:
# encrypt it
enc_noise = encrypt(noise, cc, keys, mode="COL_MAJOR")

In [30]:
enc_noise.shape, enc_noise.original_shape

((4096, 1), (4096,))

In [31]:
enc_noise.order

<ArrayEncodingType.COL_MAJOR: 1>

In [32]:
start, end = 0, 8

In [33]:
sliced_noise = slice_vector_he(enc_noise, start, end, keys, debug=True)

> /tmp/ipykernel_1747013/1135424078.py(35)slice_vector_he()
     33         import pdb; pdb.set_trace()
     34 
---> 35     enc_selection = onp.array(
     36         cc=enc_vec.data.GetCryptoContext(),
     37         data=selection_matrix,



ipdb>  c


In [34]:
sliced_noise.decrypt(keys.secretKey, unpack_type="original")

array([ 0.41461546,  0.86472529,  0.37381813,  0.97874106, -0.96869407,
       -0.80345936,  0.99081797, -0.69676019])

In [35]:
noise[start:end]

array([ 0.41461546,  0.86472529,  0.37381813,  0.97874106, -0.96869407,
       -0.80345936,  0.99081797, -0.69676019])

In [36]:
sliced_noise.order

<ArrayEncodingType.ROW_MAJOR: 0>

In [37]:
sliced_noise.shape

(8, 4096)

In [38]:
# data matrix
data_matrix = np.arange(1, 5*12+1).reshape(5, 12)

In [40]:
data_matrix.sum(axis=0)

array([125, 130, 135, 140, 145, 150, 155, 160, 165, 170, 175, 180])

In [42]:
enc_data = encrypt(data_matrix, cc, keys)
enc_data.extra["rowkey"] = onp.sum_row_keys(keys.secretKey, enc_data.ncols, enc_data.batch_size)

In [46]:
oneway = onp.sum(enc_data, axis=0) * (1/256)

In [47]:
oneway.decrypt(keys.secretKey, unpack_type="original")

array([125., 130., 135., 140., 145., 150., 155., 160., 165., 170., 175.,
       180.])

In [48]:
oneway.shape

(8, 16)

In [49]:
oneway.original_shape

(12,)

In [50]:
oneway_enc = encrypt(data_matrix.sum(axis=0), cc, keys, mode="COL_MAJOR")

In [51]:
oneway_enc.shape

(16, 1)

In [52]:
oneway_enc.original_shape

(12,)

In [53]:
oneway.order

<ArrayEncodingType.COL_MAJOR: 1>